# SOPR Capitulation Signal - Advanced Exit Strategies

## Exit Strategies to Test
1. **Profit Target** - Exit at +X% gain
2. **Trailing Stop** - Lock in gains, exit on pullback
3. **SOPR Recovery** - Exit when SOPR crosses above 1.02
4. **Stop Loss** - Exit at -X% loss
5. **Combo** - First of: profit target OR stop loss OR max hold

## Better Visualization
- Zoomed charts for specific periods
- Clear trade annotations

In [ ]:
import pandas as pd
import numpy as np
import vectorbt as vbt
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

vbt.settings.plotting['use_widgets'] = False
vbt.settings.array_wrapper['freq'] = 'D'
vbt.settings.portfolio['init_cash'] = 100_000

print(f"VectorBT version: {vbt.__version__}")

In [ ]:
# Load data
DATA_DIR = Path("../data/daily")

sopr = pd.read_parquet(DATA_DIR / "sopr.parquet").rename(columns={"value": "sopr"}).set_index("time")
sopr_sth = pd.read_parquet(DATA_DIR / "sopr_sth.parquet").rename(columns={"value": "sopr_sth"}).set_index("time")
price = pd.read_parquet(DATA_DIR / "price.parquet").rename(columns={"value": "price"}).set_index("time")

df = sopr.join(sopr_sth, how='inner').join(price, how='inner').sort_index()
df = df[df.index >= '2018-12-15']  # 2019+

close = df['price']
print(f"Data: {len(df)} rows, {df.index.min().date()} to {df.index.max().date()}")

In [ ]:
# Entry signal: First day of double capitulation
both_below_1 = (df['sopr'] < 1) & (df['sopr_sth'] < 1)
entries = both_below_1 & ~both_below_1.shift(1).fillna(False)

print(f"Entry signals: {entries.sum()}")

---
## 1. Better Visualization

Let's look at specific time periods with clear annotations.

In [ ]:
def plot_period(start_date, end_date, title):
    """Plot a specific time period with entry signals clearly marked."""
    mask = (close.index >= start_date) & (close.index <= end_date)
    period_close = close[mask]
    period_entries = entries[mask]
    period_sopr = df.loc[mask, 'sopr']
    period_sopr_sth = df.loc[mask, 'sopr_sth']
    
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        row_heights=[0.6, 0.4],
                        subplot_titles=[f'BTC Price - {title}', 'SOPR Metrics'])
    
    # Price
    fig.add_trace(go.Scatter(x=period_close.index, y=period_close, 
                             name='Price', line=dict(color='blue', width=2)), row=1, col=1)
    
    # Entry markers with annotations
    entry_dates = period_entries[period_entries].index
    for date in entry_dates:
        entry_price = period_close.loc[date]
        fig.add_trace(go.Scatter(
            x=[date], y=[entry_price],
            mode='markers+text',
            marker=dict(symbol='triangle-up', size=20, color='green', 
                       line=dict(width=2, color='darkgreen')),
            text=[f'BUY<br>${entry_price:,.0f}'],
            textposition='bottom center',
            textfont=dict(size=10, color='green'),
            showlegend=False
        ), row=1, col=1)
    
    # SOPR
    fig.add_trace(go.Scatter(x=period_sopr.index, y=period_sopr, 
                             name='SOPR', line=dict(color='orange')), row=2, col=1)
    fig.add_trace(go.Scatter(x=period_sopr_sth.index, y=period_sopr_sth, 
                             name='STH SOPR', line=dict(color='purple')), row=2, col=1)
    fig.add_hline(y=1, line_dash='dash', line_color='red', row=2, col=1)
    
    # Shade capitulation zones
    both_below = (period_sopr < 1) & (period_sopr_sth < 1)
    for i in range(len(both_below)):
        if both_below.iloc[i]:
            fig.add_vrect(x0=both_below.index[i], x1=both_below.index[min(i+1, len(both_below)-1)],
                         fillcolor='red', opacity=0.1, line_width=0, row=1, col=1)
    
    fig.update_layout(height=700, title_text=title)
    fig.update_yaxes(title_text='Price ($)', row=1, col=1)
    fig.update_yaxes(title_text='SOPR', row=2, col=1)
    return fig

In [ ]:
# COVID Crash - March 2020
fig = plot_period('2020-01-01', '2020-06-01', 'COVID Crash (Mar 2020)')
fig.show()

In [ ]:
# 2022 Bear Market
fig = plot_period('2022-01-01', '2022-12-31', '2022 Bear Market')
fig.show()

In [ ]:
# Recent - 2024
fig = plot_period('2024-01-01', '2025-01-01', '2024')
fig.show()

---
## 2. Custom Exit Strategy Function

Build a flexible backtester that can test different exit rules.

In [ ]:
def backtest_with_exits(
    close: pd.Series,
    entries: pd.Series,
    sopr: pd.Series = None,
    profit_target: float = None,      # e.g., 0.20 for +20%
    stop_loss: float = None,          # e.g., 0.10 for -10%
    trailing_stop: float = None,      # e.g., 0.15 for 15% from peak
    sopr_exit: float = None,          # e.g., 1.02 - exit when SOPR crosses above
    max_hold_days: int = None,        # e.g., 60 - max days to hold
    verbose: bool = False
):
    """
    Custom backtest with flexible exit rules.
    Exit triggers on FIRST condition met.
    """
    trades = []
    
    entry_indices = entries[entries].index.tolist()
    
    i = 0
    while i < len(entry_indices):
        entry_date = entry_indices[i]
        entry_idx = close.index.get_loc(entry_date)
        entry_price = close.iloc[entry_idx]
        
        # Track trade
        peak_price = entry_price
        exit_date = None
        exit_price = None
        exit_reason = None
        
        # Scan forward from entry
        for j in range(entry_idx + 1, len(close)):
            current_date = close.index[j]
            current_price = close.iloc[j]
            days_held = j - entry_idx
            
            # Update peak for trailing stop
            if current_price > peak_price:
                peak_price = current_price
            
            pnl_pct = (current_price - entry_price) / entry_price
            drawdown_from_peak = (peak_price - current_price) / peak_price
            
            # Check exit conditions (order matters - first one wins)
            
            # 1. Stop Loss
            if stop_loss and pnl_pct <= -stop_loss:
                exit_date = current_date
                exit_price = current_price
                exit_reason = f'stop_loss_{stop_loss*100:.0f}%'
                break
            
            # 2. Profit Target
            if profit_target and pnl_pct >= profit_target:
                exit_date = current_date
                exit_price = current_price
                exit_reason = f'profit_target_{profit_target*100:.0f}%'
                break
            
            # 3. Trailing Stop
            if trailing_stop and drawdown_from_peak >= trailing_stop:
                exit_date = current_date
                exit_price = current_price
                exit_reason = f'trailing_stop_{trailing_stop*100:.0f}%'
                break
            
            # 4. SOPR Recovery Exit
            if sopr_exit and sopr is not None:
                if current_date in sopr.index:
                    current_sopr = sopr.loc[current_date]
                    if current_sopr >= sopr_exit:
                        exit_date = current_date
                        exit_price = current_price
                        exit_reason = f'sopr_above_{sopr_exit}'
                        break
            
            # 5. Max Hold Days
            if max_hold_days and days_held >= max_hold_days:
                exit_date = current_date
                exit_price = current_price
                exit_reason = f'max_hold_{max_hold_days}d'
                break
        
        # If no exit triggered, exit at end
        if exit_date is None:
            exit_date = close.index[-1]
            exit_price = close.iloc[-1]
            exit_reason = 'end_of_data'
        
        # Record trade
        pnl = (exit_price - entry_price) / entry_price
        trades.append({
            'entry_date': entry_date,
            'entry_price': entry_price,
            'exit_date': exit_date,
            'exit_price': exit_price,
            'pnl_pct': pnl,
            'days_held': (exit_date - entry_date).days,
            'exit_reason': exit_reason
        })
        
        if verbose:
            print(f"{entry_date.date()} → {exit_date.date()}: {pnl*100:+.1f}% ({exit_reason})")
        
        # Skip entries that occur during this trade
        while i < len(entry_indices) and entry_indices[i] <= exit_date:
            i += 1
    
    return pd.DataFrame(trades)

---
## 3. Test Exit Strategies

In [ ]:
# Strategy configurations to test
strategies = {
    # Profit targets only
    'PT_10%': {'profit_target': 0.10, 'max_hold_days': 90},
    'PT_20%': {'profit_target': 0.20, 'max_hold_days': 90},
    'PT_30%': {'profit_target': 0.30, 'max_hold_days': 90},
    
    # Stop loss only
    'SL_10%': {'stop_loss': 0.10, 'max_hold_days': 90},
    'SL_15%': {'stop_loss': 0.15, 'max_hold_days': 90},
    
    # Trailing stop
    'Trail_15%': {'trailing_stop': 0.15, 'max_hold_days': 90},
    'Trail_20%': {'trailing_stop': 0.20, 'max_hold_days': 90},
    
    # SOPR recovery
    'SOPR_1.02': {'sopr_exit': 1.02, 'max_hold_days': 90},
    'SOPR_1.05': {'sopr_exit': 1.05, 'max_hold_days': 90},
    
    # Combos - these are the interesting ones
    'PT20_SL10': {'profit_target': 0.20, 'stop_loss': 0.10, 'max_hold_days': 90},
    'PT30_SL15': {'profit_target': 0.30, 'stop_loss': 0.15, 'max_hold_days': 90},
    'PT20_Trail15': {'profit_target': 0.20, 'trailing_stop': 0.15, 'max_hold_days': 90},
    'Trail20_SL10': {'trailing_stop': 0.20, 'stop_loss': 0.10, 'max_hold_days': 90},
    'SOPR_SL10': {'sopr_exit': 1.02, 'stop_loss': 0.10, 'max_hold_days': 90},
    'SOPR_PT20_SL10': {'sopr_exit': 1.02, 'profit_target': 0.20, 'stop_loss': 0.10, 'max_hold_days': 90},
}

In [ ]:
# Run all strategies
results = []

for name, params in strategies.items():
    trades_df = backtest_with_exits(
        close=close,
        entries=entries,
        sopr=df['sopr'],
        **params
    )
    
    if len(trades_df) > 0:
        # Calculate metrics
        total_return = (1 + trades_df['pnl_pct']).prod() - 1
        win_rate = (trades_df['pnl_pct'] > 0).mean()
        avg_win = trades_df[trades_df['pnl_pct'] > 0]['pnl_pct'].mean() if (trades_df['pnl_pct'] > 0).any() else 0
        avg_loss = trades_df[trades_df['pnl_pct'] <= 0]['pnl_pct'].mean() if (trades_df['pnl_pct'] <= 0).any() else 0
        avg_days = trades_df['days_held'].mean()
        
        # Profit factor
        gross_profit = trades_df[trades_df['pnl_pct'] > 0]['pnl_pct'].sum()
        gross_loss = abs(trades_df[trades_df['pnl_pct'] <= 0]['pnl_pct'].sum())
        profit_factor = gross_profit / gross_loss if gross_loss > 0 else np.inf
        
        results.append({
            'strategy': name,
            'n_trades': len(trades_df),
            'total_return': total_return,
            'win_rate': win_rate,
            'avg_win': avg_win,
            'avg_loss': avg_loss,
            'profit_factor': profit_factor,
            'avg_days': avg_days
        })

results_df = pd.DataFrame(results).sort_values('total_return', ascending=False)

print("STRATEGY COMPARISON")
print("="*100)
print(results_df.to_string(index=False))

In [ ]:
# Visualize results
fig = make_subplots(rows=2, cols=2,
                    subplot_titles=['Total Return', 'Win Rate', 'Profit Factor', 'Avg Days Held'])

# Sort by total return for display
plot_df = results_df.sort_values('total_return', ascending=True)

# Colors based on return
colors = ['green' if x > 0 else 'red' for x in plot_df['total_return']]

fig.add_trace(go.Bar(y=plot_df['strategy'], x=plot_df['total_return']*100, 
                     orientation='h', marker_color=colors), row=1, col=1)
fig.add_trace(go.Bar(y=plot_df['strategy'], x=plot_df['win_rate']*100, 
                     orientation='h', marker_color='steelblue'), row=1, col=2)
fig.add_trace(go.Bar(y=plot_df['strategy'], x=plot_df['profit_factor'].clip(upper=5), 
                     orientation='h', marker_color='purple'), row=2, col=1)
fig.add_trace(go.Bar(y=plot_df['strategy'], x=plot_df['avg_days'], 
                     orientation='h', marker_color='orange'), row=2, col=2)

fig.add_vline(x=0, line_dash='dash', row=1, col=1)
fig.add_vline(x=50, line_dash='dash', row=1, col=2)
fig.add_vline(x=1, line_dash='dash', row=2, col=1)

fig.update_layout(height=800, showlegend=False, title_text='Exit Strategy Comparison')
fig.show()

In [ ]:
# Best strategy details
best_name = results_df.iloc[0]['strategy']
best_params = strategies[best_name]

print(f"\n{'='*60}")
print(f"BEST STRATEGY: {best_name}")
print(f"{'='*60}")
print(f"Parameters: {best_params}")
print(f"\nMetrics:")
for col in results_df.columns:
    if col != 'strategy':
        val = results_df.iloc[0][col]
        if 'return' in col or 'rate' in col or 'win' in col or 'loss' in col:
            print(f"  {col}: {val*100:.1f}%")
        else:
            print(f"  {col}: {val:.2f}")

---
## 4. Detailed Analysis of Best Strategy

In [ ]:
# Run best strategy with details
trades_best = backtest_with_exits(
    close=close,
    entries=entries,
    sopr=df['sopr'],
    **best_params,
    verbose=False
)

print(f"\nTRADE DETAILS - {best_name}")
print("="*90)
trades_best['entry_date'] = pd.to_datetime(trades_best['entry_date']).dt.date
trades_best['exit_date'] = pd.to_datetime(trades_best['exit_date']).dt.date
trades_best['pnl_pct'] = (trades_best['pnl_pct'] * 100).round(1)
trades_best['entry_price'] = trades_best['entry_price'].round(0)
trades_best['exit_price'] = trades_best['exit_price'].round(0)

print(trades_best.to_string(index=False))

In [ ]:
# Exit reason breakdown
print("\nEXIT REASON BREAKDOWN")
print("="*40)
exit_counts = trades_best['exit_reason'].value_counts()
for reason, count in exit_counts.items():
    pct = count / len(trades_best) * 100
    avg_pnl = trades_best[trades_best['exit_reason'] == reason]['pnl_pct'].mean()
    print(f"{reason}: {count} trades ({pct:.0f}%) - Avg PnL: {avg_pnl:+.1f}%")

In [ ]:
# Reload trades with datetime for plotting
trades_plot = backtest_with_exits(
    close=close,
    entries=entries,
    sopr=df['sopr'],
    **best_params
)

# Plot trades on price chart
fig = go.Figure()

# Price
fig.add_trace(go.Scatter(x=close.index, y=close, name='BTC Price', 
                         line=dict(color='lightblue', width=1)))

# Draw each trade
for _, trade in trades_plot.iterrows():
    color = 'green' if trade['pnl_pct'] > 0 else 'red'
    
    # Trade line
    fig.add_trace(go.Scatter(
        x=[trade['entry_date'], trade['exit_date']],
        y=[trade['entry_price'], trade['exit_price']],
        mode='lines+markers',
        line=dict(color=color, width=2),
        marker=dict(size=8),
        showlegend=False,
        hovertemplate=f"Entry: {trade['entry_date'].date()}<br>" +
                      f"Exit: {trade['exit_date'].date()}<br>" +
                      f"PnL: {trade['pnl_pct']*100:.1f}%<br>" +
                      f"Reason: {trade['exit_reason']}<extra></extra>"
    ))

fig.update_layout(
    title=f'All Trades - {best_name}<br><sup>Green=Win, Red=Loss</sup>',
    yaxis_title='Price ($)',
    yaxis_type='log',
    height=600
)
fig.show()

In [ ]:
# Equity curve
cumulative_returns = (1 + trades_plot['pnl_pct']).cumprod()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=trades_plot['exit_date'],
    y=cumulative_returns * 100000,  # Start with 100k
    mode='lines+markers',
    name='Strategy Equity'
))

# Buy and hold for comparison
bh_return = close.iloc[-1] / close.iloc[0]
fig.add_hline(y=100000 * bh_return, line_dash='dash', line_color='gray',
              annotation_text=f'Buy & Hold: ${100000*bh_return:,.0f}')
fig.add_hline(y=100000, line_dash='dot', line_color='black')

fig.update_layout(
    title=f'Equity Curve - {best_name}',
    yaxis_title='Portfolio Value ($)',
    height=500
)
fig.show()

---
## 5. Walk-Forward Validation

In [ ]:
# Walk-forward test
def walk_forward_test(strategy_params, train_days=365, test_days=90, step_days=90):
    """Walk-forward validation for a strategy."""
    wf_results = []
    
    total_days = len(close)
    n_folds = (total_days - train_days) // step_days
    
    for fold in range(n_folds):
        test_start = train_days + fold * step_days
        test_end = min(test_start + test_days, total_days)
        
        if test_end <= test_start:
            break
        
        # Get test period data
        test_close = close.iloc[test_start:test_end]
        test_entries = entries.iloc[test_start:test_end]
        test_sopr = df['sopr'].iloc[test_start:test_end]
        
        # Run strategy on test period
        trades = backtest_with_exits(
            close=test_close,
            entries=test_entries,
            sopr=test_sopr,
            **strategy_params
        )
        
        # Calculate returns
        if len(trades) > 0:
            strat_return = (1 + trades['pnl_pct']).prod() - 1
            n_trades = len(trades)
        else:
            strat_return = 0
            n_trades = 0
        
        hold_return = (test_close.iloc[-1] / test_close.iloc[0]) - 1
        
        wf_results.append({
            'fold': fold,
            'period': close.index[test_start].strftime('%Y-%m'),
            'n_trades': n_trades,
            'strat_return': strat_return,
            'hold_return': hold_return,
            'excess': strat_return - hold_return,
            'beat_hold': strat_return > hold_return
        })
    
    return pd.DataFrame(wf_results)

In [ ]:
# Run walk-forward for best strategy
wf_df = walk_forward_test(best_params)

print(f"\nWALK-FORWARD RESULTS - {best_name}")
print("="*80)

for _, row in wf_df.iterrows():
    status = '✓' if row['beat_hold'] else '✗'
    print(f"Fold {row['fold']:2d}: {row['period']} | "
          f"{row['n_trades']:2d} trades | "
          f"Strat: {row['strat_return']*100:+6.1f}% | "
          f"B&H: {row['hold_return']*100:+6.1f}% | {status}")

In [ ]:
# Walk-forward summary
print("\n" + "="*60)
print("WALK-FORWARD SUMMARY")
print("="*60)

wf_with_trades = wf_df[wf_df['n_trades'] > 0]

print(f"\n{'Metric':<35} {'Value':>15}")
print("-"*55)
print(f"{'Total Folds':<35} {len(wf_df):>15}")
print(f"{'Folds with Trades':<35} {len(wf_with_trades):>15}")
print(f"{'Avg Strategy Return':<35} {wf_df['strat_return'].mean()*100:>14.1f}%")
print(f"{'Avg Buy&Hold Return':<35} {wf_df['hold_return'].mean()*100:>14.1f}%")
print(f"{'Avg Excess Return':<35} {wf_df['excess'].mean()*100:>+14.1f}%")
print(f"{'Beat Buy&Hold Rate':<35} {wf_df['beat_hold'].mean()*100:>14.1f}%")

if wf_df['beat_hold'].mean() > 0.5:
    verdict = "✓ STRATEGY WORKS"
else:
    verdict = "✗ STRATEGY DOESN'T BEAT BUY & HOLD"

print(f"\n🎯 VERDICT: {verdict}")

---
## 6. Final Summary

In [ ]:
print("\n" + "="*70)
print("SOPR CAPITULATION SIGNAL - FINAL RESULTS")
print("="*70)

print(f"\n📊 SIGNAL")
print(f"   Entry: When SOPR < 1 AND STH SOPR < 1")
print(f"   Exit Strategy: {best_name}")
print(f"   Parameters: {best_params}")

print(f"\n📈 IN-SAMPLE PERFORMANCE")
best_row = results_df.iloc[0]
print(f"   Total Return: {best_row['total_return']*100:.1f}%")
print(f"   Win Rate: {best_row['win_rate']*100:.0f}%")
print(f"   Profit Factor: {best_row['profit_factor']:.2f}")
print(f"   Avg Days Held: {best_row['avg_days']:.0f}")

print(f"\n🔍 WALK-FORWARD VALIDATION")
print(f"   Beat Buy&Hold: {wf_df['beat_hold'].mean()*100:.0f}%")
print(f"   Avg Excess Return: {wf_df['excess'].mean()*100:+.1f}%")

print("\n" + "="*70)

In [ ]:
# Save results
import json

final_results = {
    'signal': 'double_capitulation',
    'entry_condition': 'SOPR < 1 AND STH_SOPR < 1',
    'best_exit_strategy': best_name,
    'exit_params': best_params,
    'in_sample': {
        'total_return': float(best_row['total_return']),
        'win_rate': float(best_row['win_rate']),
        'profit_factor': float(best_row['profit_factor']),
        'n_trades': int(best_row['n_trades']),
        'avg_days': float(best_row['avg_days'])
    },
    'walk_forward': {
        'beat_hold_pct': float(wf_df['beat_hold'].mean()),
        'avg_excess_return': float(wf_df['excess'].mean()),
        'n_folds': len(wf_df)
    },
    'all_strategies_tested': results_df.to_dict('records')
}

with open('../data/sopr_advanced_exit_results.json', 'w') as f:
    json.dump(final_results, f, indent=2, default=str)

print("Saved to ../data/sopr_advanced_exit_results.json")